In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config.settings import (
    FEATURE_NAMES,
    TARGET_COLUMN,
    TARGET_CLASSES,
    get_raw_dataset_path,
    PROCESSED_DATASET_DIR,
)
from src.data.dataset_inspector import compute_file_sha256
from src.preprocessing.data_cleaner import clean_dry_beans_dataset
from src.preprocessing.data_validator import validate_dataset

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Phase 3 environment initialized.')

Phase 3 environment initialized.


In [2]:
raw_path = get_raw_dataset_path()
raw_sha = compute_file_sha256(raw_path)
df_raw = pd.read_excel(raw_path)

print(f'Raw Dataset Path: {raw_path}')
print(f'SHA-256 Checksum: {raw_sha}')
print(f'Total Raw Observations: {len(df_raw):,}')
print(f'Total Columns: {len(df_raw.columns)}')
print(f'Target Column: {TARGET_COLUMN} (Present: {TARGET_COLUMN in df_raw.columns})')
print(f'Unique Classes: {sorted(df_raw[TARGET_COLUMN].unique().tolist())}')

Raw Dataset Path: C:\Users\A.VARUN KUMAR REDDY\OneDrive\Desktop\AGRO\DryBeanDataset\Dry_Bean_Dataset.xlsx
SHA-256 Checksum: a9efa69741c6c5d95167c962a13da23a456f7e940402a5add12639790e01f714
Total Raw Observations: 13,611
Total Columns: 17
Target Column: Class (Present: True)
Unique Classes: ['BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SEKER', 'SIRA']


In [3]:
missing_count = int(df_raw.isnull().sum().sum())
exact_dups = int(df_raw.duplicated().sum())
negative_vals = int((df_raw[FEATURE_NAMES] < 0).sum().sum())
inf_vals = int(np.isinf(df_raw[FEATURE_NAMES]).sum().sum())

quality_df = pd.DataFrame([
    {'Metric': 'Missing Cells', 'Count': missing_count, 'Pct': 0.0},
    {'Metric': 'Exact Duplicate Rows', 'Count': exact_dups, 'Pct': round(exact_dups / len(df_raw) * 100, 3)},
    {'Metric': 'Negative Feature Values', 'Count': negative_vals, 'Pct': 0.0},
    {'Metric': 'Infinite Feature Values', 'Count': inf_vals, 'Pct': 0.0},
])
display(quality_df)

,Metric,Count,Pct
0,Missing Cells,0,0.0
1,Exact Duplicate Rows,68,0.5
2,Negative Feature Values,0,0.0
3,Infinite Feature Values,0,0.0


In [4]:
cleaned_df, report = clean_dry_beans_dataset()
print(f'Original Rows: {report.raw_row_count:,}')
print(f'Duplicates Removed: {report.exact_duplicates_removed}')
print(f'Cleaned Working Rows: {report.cleaned_row_count:,}')
print(f'Cleaned File Saved: {report.cleaned_file_path}')

Original Rows: 13,611
Duplicates Removed: 68
Cleaned Working Rows: 13,543
Cleaned File Saved: C:\Users\A.VARUN KUMAR REDDY\OneDrive\Desktop\AGRO\dataset\processed\cleaned_dry_beans.csv


In [5]:
classes = sorted(report.class_distribution_before.keys())
comp_df = pd.DataFrame({
    'Class': classes,
    'Raw Count': [report.class_distribution_before[c] for c in classes],
    'Cleaned Count': [report.class_distribution_after[c] for c in classes],
    'Duplicates Removed': [report.class_distribution_before[c] - report.class_distribution_after[c] for c in classes],
})
comp_df['Raw Pct'] = (comp_df['Raw Count'] / report.raw_row_count * 100).round(2)
comp_df['Cleaned Pct'] = (comp_df['Cleaned Count'] / report.cleaned_row_count * 100).round(2)
display(comp_df)

,Class,Raw Count,Cleaned Count,Duplicates Removed,Raw Pct,Cleaned Pct
0,BARBUNYA,1322,1322,0,9.71,9.76
1,BOMBAY,522,522,0,3.84,3.85
2,CALI,1630,1630,0,11.98,12.04
3,DERMASON,3546,3546,0,26.05,26.18
4,HOROZ,1928,1860,68,14.17,13.73
5,SEKER,2027,2027,0,14.89,14.97
6,SIRA,2636,2636,0,19.37,19.46


In [6]:
validation_res = validate_dataset(cleaned_df)
print('Validation Result:')
for k, v in validation_res.items():
    print(f' - {k}: {v}')

Validation Result:
 - is_valid: True
 - row_count: 13543
 - column_count: 17
 - target_classes_present: ['BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SEKER', 'SIRA']
